# Training a tuned-lens

cd /media/am/AM/mi-lens
PYTHONPATH=/media/am/AM/mi-lens/lenses/tuned_logit_lens \
python -m tuned_lens train \
  --data.name /media/am/AM/mi-lens/tmp/wikitext_100_for_tuned_lens.jsonl \
  --model.name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --precision bfloat16 \
  --max_seq_len 128 \
  --tokens_per_step 128 \
  --num_steps 100 \
  --loss KL \
  --token_shift 0 \
  --output /media/am/AM/mi-lens/tmp/tuned_lens_tinyllama_wikitext100_v2

(no checkpoint steps saved)

In [ ]:
import json
import sys
from pathlib import Path
import torch

ROOT = Path("/media/am/AM/mi-lens")
sys.path.insert(0, str(ROOT / "lenses" / "jacobian_lens"))

from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=100)

out_path = ROOT / "tmp" / "wikitext_100_for_tuned_lens.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("w", encoding="utf-8") as f:
    for text in prompts:
        f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")

print(out_path)
print("n_prompts:", len(prompts))

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

ROOT = Path("/media/am/AM/mi-lens")
TMP = ROOT / "tmp"
TMP.mkdir(parents=True, exist_ok=True)

# Make the vendored packages importable
sys.path.insert(0, str(ROOT / "lenses" / "jacobian_lens"))
sys.path.insert(0, str(ROOT / "lenses" / "tuned_logit_lens"))

from jlens.examples import load_wikitext_prompts

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
JSONL_PATH = TMP / "wikitext_100_for_tuned_lens.jsonl"
OUTPUT_DIR = TMP / "tuned_lens_tinyllama_wikitext100_v2"

# Same data source as the J-lens walkthrough example
prompts = load_wikitext_prompts(n_prompts=100)

with JSONL_PATH.open("w", encoding="utf-8") as f:
    for text in prompts:
        f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")

print("JSONL:", JSONL_PATH)
print("Exists:", JSONL_PATH.exists())
print("Prompts:", len(prompts))

env = os.environ.copy()
existing = env.get("PYTHONPATH", "")
vendor_path = str(ROOT / "lenses" / "tuned_logit_lens")
env["PYTHONPATH"] = vendor_path if not existing else vendor_path + os.pathsep + existing

cmd = [
    sys.executable,
    "-m",
    "tuned_lens",
    "train",
    "--data.name",
    str(JSONL_PATH),
    "--model.name",
    MODEL_NAME,
    "--precision",
    "bfloat16",
    "--max_seq_len",
    "128",
    "--tokens_per_step",
    "128",
    "--num_steps",
    "100",
    "--loss",
    "KL",
    "--token_shift",
    "0",
    "--output",
    str(OUTPUT_DIR),
]

print("\nRunning:\n")
print(" ".join(shlex.quote(x) for x in cmd))
print()

subprocess.run(cmd, env=env, check=True)

In [ ]:
import sys
from pathlib import Path
import torch
import transformers

ROOT = Path("/media/am/AM/mi-lens")
sys.path.insert(0, str(ROOT / "lenses" / "tuned_logit_lens"))

from tuned_lens import TunedLens

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).cpu()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

tuned_lens = TunedLens.from_model_and_pretrained(
    hf_model,
    str(ROOT / "tmp" / "tuned_lens_tinyllama_wikitext100_v2"),
)
tuned_lens